## Advanced_CI_w6
### CH11

### 11.1 불응
### 11.2 잠재적 결과 확장

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("prime_card.csv")

df.head()

,age,income,credit_score,prime_elegible,prime_card,pv,tau,categ
0,37.7,9687.0,822.0,0,0,4913.79,700.0,complier
1,46.0,13731.0,190.0,0,0,5637.66,200.0,never-taker
2,43.1,2839.0,214.0,1,1,2410.45,700.0,complier
3,36.0,1206.0,318.0,1,1,1363.06,700.0,complier
4,39.7,4095.0,430.0,0,0,2189.80,700.0,complier


In [3]:
import statsmodels.formula.api as smf

m = smf.ols("pv~prime_elegible", data = df).fit()
m.summary().tables[1]

,coef,std err,t,P>|t|,[0.025,0.975]
Intercept,2498.3618,24.327,102.701,0.000,2450.677,2546.047
prime_elegible,321.3880,34.321,9.364,0.000,254.113,388.663


In [4]:
float(df["tau"].mean())


413.45

In [5]:
m = smf.ols("pv~prime_card", data = df).fit()
m.summary().tables[1]

,coef,std err,t,P>|t|,[0.025,0.975]
Intercept,2534.4947,19.239,131.740,0.000,2496.783,2572.206
prime_card,588.1388,41.676,14.112,0.000,506.446,669.831


### 11.3 도구변수 식별 
### 11.4 1단계

In [6]:
first_stage = smf.ols("prime_card~prime_elegible", data = df).fit()
first_stage.summary().tables[1]

,coef,std err,t,P>|t|,[0.025,0.975]
Intercept,1.415e-15,0.005,2.85e-13,1.000,-0.010,0.010
prime_elegible,0.4242,0.007,60.536,0.000,0.410,0.438


In [7]:
df.groupby("categ").size()/len(df)

categ
complier       0.4269
never-taker    0.5731
dtype: float64

### 11.5 2단계

In [8]:
red_form = smf.ols("pv~prime_elegible",data=df).fit()
red_form.summary().tables[1]

,coef,std err,t,P>|t|,[0.025,0.975]
Intercept,2498.3618,24.327,102.701,0.000,2450.677,2546.047
prime_elegible,321.3880,34.321,9.364,0.000,254.113,388.663


In [9]:
late = (red_form.params["prime_elegible"]/first_stage.params["prime_elegible"])
float(late)

757.6973795343838

In [10]:
df.groupby("categ")["tau"].mean()

categ
complier       700.0
never-taker    200.0
Name: tau, dtype: float64

### 11.6 2단계 최소제곱법

In [11]:
iv_regr = smf.ols("pv~prime_card", data = df.assign(prime_card = first_stage.fittedvalues)).fit() # hat
iv_regr.summary().tables[1]

,coef,std err,t,P>|t|,[0.025,0.975]
Intercept,2498.3618,24.327,102.701,0.000,2450.677,2546.047
prime_card,757.6974,80.914,9.364,0.000,599.091,916.304


### 11.7 표준오차

In [12]:
Z = df["prime_elegible"]
T = df["prime_card"]
n = len(df)

# iv_regr.resid와 차이가 있음
e_iv = df["pv"] - iv_regr.predict(df)
compliance = np.cov(T,Z)[0,1]/Z.var()

se = np.std(e_iv)/(compliance*np.std(Z)*np.sqrt(n))

print("SE IV:", se)
print("95% CI:", [float(late-2*se), float(late+2*se)])

SE IV: 80.5286102614222
95% CI: [596.6401590115394, 918.7546000572282]


In [13]:
%pip install linearmodels

Note: you may need to restart the kernel to use updated packages.


In [14]:
from linearmodels import IV2SLS

formula = 'pv~1+[prime_card ~ prime_elegible]'
iv_model = IV2SLS.from_formula(formula,df).fit(cov_type="unadjusted")

iv_model.summary.tables[1]

,Parameter,Std. Err.,T-stat,P-value,Lower CI,Upper CI
Intercept,2498.4,24.211,103.19,0.0000,2450.9,2545.8
prime_card,757.70,80.529,9.4090,0.0000,599.86,915.53


### 11.8 통제변수와 도구변수 추가

In [15]:
formula = 'pv ~ 1 + [prime_card ~ prime_elegible + credit_score]'
iv_model = IV2SLS.from_formula(formula, df).fit()

iv_model.summary.tables[1]

,Parameter,Std. Err.,T-stat,P-value,Lower CI,Upper CI
Intercept,2519.4,21.168,119.02,0.0000,2477.9,2560.9
prime_card,659.04,58.089,11.345,0.0000,545.19,772.90


In [16]:
formula = '''pv ~ 1
+ [prime_card ~ prime_elegible + credit_score]
+ income + age'''

iv_model = IV2SLS.from_formula(formula, df).fit(cov_type = "unadjusted")
iv_model.summary.tables[1]

,Parameter,Std. Err.,T-stat,P-value,Lower CI,Upper CI
Intercept,210.62,37.605,5.6008,0.0000,136.91,284.32
income,0.3998,0.0008,471.04,0.0000,0.3981,0.4014
age,9.7444,0.8873,10.982,0.0000,8.0053,11.483
prime_card,693.12,12.165,56.978,0.0000,669.28,716.96


In [17]:
formula_1st = "prime_card ~ prime_elegible + credit_score + income + age"
first_stage = smf.ols(formula_1st, data = df).fit()

iv_model = smf.ols(
    "pv~prime_card + income + age",
    data = df.assign(prime_card = first_stage.fittedvalues)).fit()

iv_model.summary().tables[1]

,coef,std err,t,P>|t|,[0.025,0.975]
Intercept,210.6177,40.832,5.158,0.000,130.578,290.657
prime_card,693.1207,13.209,52.474,0.000,667.229,719.013
income,0.3998,0.001,433.806,0.000,0.398,0.402
age,9.7444,0.963,10.114,0.000,7.856,11.633


In [18]:
Z = df[["prime_elegible", "credit_score", "income", "age"]].values
X = df[["prime_card", "income", "age"]].values
Y = df[["pv"]].values

def add_intercept(x):
    return np.concatenate([np.ones((x.shape[0], 1)), x], axis=1)

Z_ = add_intercept(Z)
X_ = add_intercept(X)

# Z_.dot(...) 사전 연산
# NxN 행렬
X_hat = Z_.dot(np.linalg.inv(Z_.T.dot(Z_)).dot(Z_.T).dot(X_))

b_iv = np.linalg.inv(X_hat.T.dot(X_hat)).dot(X_hat.T).dot(Y)
b_iv[1]

array([693.12072518])

In [19]:
e_hat_iv = Y - X_.dot(b_iv)

var = e_hat_iv.var()*np.diag(np.linalg.inv(X_hat.T.dot(X_hat)))
float(np.sqrt(var[1]))

12.16469439503136

In [20]:
t_tilde = smf.ols("prime_card ~ income + age", data = df).fit().resid

float(e_hat_iv.std()/(t_tilde.std()*np.sqrt(n*first_stage.rsquared)))


12.156252763191087

### 11.9 불연속 설계

In [21]:
df_dd = pd.read_csv("prime_card_discontinuity.csv")
df_dd.head()

,balance,prime_card,pv,tau,categ
0,12100.0,1,356.472,300.0,always-takers
1,4400.0,1,268.172,300.0,always-takers
2,4600.0,1,668.896,300.0,always-takers
3,3500.0,1,428.094,300.0,always-takers
4,12700.0,1,1619.793,700.0,complier


In [22]:
m = smf.ols(f"pv~balance*I(balance>0)", df_dd.assign(balance=lambda d: d["balance"]-5000)).fit()
m.summary().tables[1]

,coef,std err,t,P>|t|,[0.025,0.975]
Intercept,559.2980,8.395,66.621,0.000,542.843,575.753
I(balance > 0)[T.True],261.0699,10.128,25.777,0.000,241.218,280.922
balance,0.0616,0.005,11.892,0.000,0.051,0.072
balance:I(balance > 0)[T.True],-0.0187,0.005,-3.488,0.000,-0.029,-0.008


In [25]:
def rdd_iv(data, y, t, r, cutoff):
    centered_df = data.assign(**{r: data[r]-cutoff})
    compliance = smf.ols(f"{t}~{r}*I({r}>0)", centered_df).fit()
    itte = smf.ols(f"{y}~{r}*I({r}>0)", centered_df).fit()

    param = f"I({r} > 0)[T.True]"
    return itte.params[param]/compliance.params[param]

float(rdd_iv(df_dd, y="pv", t="prime_card", r="balance", cutoff=5000))

732.8534752298938

In [29]:
float(df_dd
.round({"balance":-2}) # 백의단위
.query("balance==5000 & categ=='complier'")["tau"].mean())

700.0